# [7] Trabalho Final - Análise dos Resultados

**Equipe:** Sem Título

**Integrantes:**

- Kayky de Brito dos Santos
- André Marques da Silva
- Rafael de Souza Coelho

**Data de publicação:** 15 de agosto de 2026

**Título do trabalho:** Programa de Reconhecimento de Valores de Cédulas

## 1. Introdução

Esta é a sétima etapa do trabalho final: a **análise dos resultados**. As etapas anteriores definiram o roteiro de testes (etapa 5) e registraram a aplicação desse roteiro com voluntários (etapa 6). Aqui fechamos o ciclo: pegamos os números crus das fichas e da enquete de opinião e respondemos, com eles, se o sistema entrega o que a modelagem funcional prometeu.

A sessão de testes voluntários ocorreu em **10 de agosto de 2026**, com **8 participantes** de fora da equipe. Cada voluntário executou uma sequência de tarefas do roteiro com cédulas reais, teve o resultado de cada tarefa anotado por um integrante da equipe e, ao final, preencheu uma ficha de opinião em papel. As fichas foram transcritas para a planilha analisada neste documento.

Três blocos de evidência são analisados aqui:

1. **Métricas objetivas:** taxa de acerto por voluntário e tempo médio até a resposta falada, anotados durante as tarefas.
2. **Usabilidade percebida:** as questões Q1 a Q11 da enquete, sendo Q1 a Q10 o questionário SUS (*System Usability Scale*) e Q11 uma questão extra sobre interatividade.
3. **Feedback aberto:** Q12 a Q17, que cobrem o que mais e o que menos agradou, sugestões e a compreensão do voluntário sobre objetivo, experimentos e resultados.

As filmagens das sessões, previstas no roteiro, são referenciadas na seção 5.

## 2. Carga dos Dados

Os dados vêm de dois arquivos em `trabalho-final/resources`: a transcrição das fichas de opinião (CSV exportado da planilha) e o texto das questões. A leitura usa apenas a biblioteca padrão e o NumPy, sem dependências adicionais no projeto.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

RECURSOS = Path("resources")
ARQUIVO_ENQUETE = RECURSOS / "Enquete de Opinião - CV26.2 - Grupo 8 - Opinioes.csv"
ARQUIVO_QUESTOES = RECURSOS / "enquete_questoes.txt"

with ARQUIVO_ENQUETE.open(encoding="utf-8") as f:
    linhas = [linha for linha in csv.reader(f)]

# As três primeiras linhas são o cabeçalho descritivo da planilha; a quarta traz os rótulos.
rotulos = linhas[3]
respostas = [
    dict(zip(rotulos, linha))
    for linha in linhas[4:]
    if linha and linha[0] and linha[0] != "Média"  # a última linha é a média da planilha
]

with ARQUIVO_QUESTOES.open(encoding="utf-8") as f:
    questoes = {
        codigo: texto
        for codigo, texto in (l.rstrip("\n").split("\t") for l in f if "\t" in l)
        if codigo != "Questão"
    }

nomes = [r["Nome completo"] for r in respostas]
print(f"{len(respostas)} voluntários: {', '.join(nomes)}")
print(f"{len(questoes)} questões carregadas (Q1 a Q{len(questoes)})")

In [ ]:
def numero(texto):
    """Converte os números da planilha, que usam vírgula decimal."""
    return float(texto.replace(",", "."))


def fracao(texto):
    """Converte 'acertos/tarefas' em um par de inteiros."""
    acertos, tarefas = texto.split("/")
    return int(acertos), int(tarefas)


acertos = np.array([fracao(r["Precisão"])[0] for r in respostas])
tarefas = np.array([fracao(r["Precisão"])[1] for r in respostas])
precisao = acertos / tarefas
tempos = np.array([numero(r["Tempo Médio (S)"]) for r in respostas])

AZUL, LARANJA, VERDE = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA_SECUNDARIA = "#0b0b0b", "#52514e"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": TINTA_SECUNDARIA,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlecolor": TINTA,
        "font.size": 9,
        "text.color": TINTA,
        "xtick.color": TINTA_SECUNDARIA,
        "ytick.color": TINTA_SECUNDARIA,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
    }
)


def barras(valores, rotulos_y, titulo, limite, formatar, cor=AZUL, referencia=None, eixo_x=None):
    """Barras horizontais com rótulo direto em cada barra e linha de referência opcional."""
    fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(valores) + 1.5))
    y = np.arange(len(valores))[::-1]
    ax.barh(y, valores, height=0.55, color=cor)
    if referencia is not None:
        rotulo_ref, valor_ref = referencia
        ax.axvline(valor_ref, color=LARANJA, linewidth=2, linestyle="--", zorder=3)
        ax.text(valor_ref, len(valores) - 0.35, f"  {rotulo_ref}", color=LARANJA, fontsize=8, va="bottom")
    for posicao, valor in zip(y, valores):
        ax.text(
            valor + limite * 0.015, posicao, formatar(valor), va="center", fontsize=8, color=TINTA, zorder=4,
            bbox=dict(facecolor="white", edgecolor="none", boxstyle="square,pad=0.15"),
        )
    ax.set_yticks(y, rotulos_y)
    ax.set_xlim(0, limite)
    if eixo_x is not None:
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: eixo_x(v)))
    ax.set_title(titulo, loc="left", pad=12)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.xaxis.grid(True, color="#eceae5", linewidth=1)
    ax.set_axisbelow(True)
    fig.tight_layout()
    return ax


print("Dados prontos.")

## 3. Métricas Objetivas

### 3.1 Taxa de acerto

Cada tarefa executada foi classificada como correta ou incorreta pelo critério do roteiro: correta quando o sistema anunciou a denominação certa (ou permaneceu em silêncio nas tarefas de bancada vazia e de objeto que não é cédula). O número de tarefas varia entre voluntários porque nem todos completaram a lista inteira dentro da janela de 10 a 15 minutos.

In [ ]:
taxa_global = acertos.sum() / tarefas.sum()

print(f"Tarefas executadas ...... {tarefas.sum()}")
print(f"Tarefas corretas ........ {acertos.sum()}")
print(f"Taxa de acerto global ... {taxa_global:.1%}")
print(f"Média por voluntário .... {precisao.mean():.1%} (desvio padrão {precisao.std(ddof=1):.1%})")
print(f"Pior e melhor sessão .... {precisao.min():.0%} e {precisao.max():.0%}")

barras(
    precisao,
    [f"{nome}  ({a}/{t})" for nome, a, t in zip(nomes, acertos, tarefas)],
    "Taxa de acerto por voluntário",
    limite=1.15,
    formatar=lambda v: f"{v:.0%}",
    eixo_x=lambda v: f"{v:.0%}",
    referencia=(f"global {taxa_global:.0%}", taxa_global),
)
plt.show()

**Leitura.** O sistema acertou **35 das 42 tarefas** executadas, uma taxa global de **83,3%**. Duas sessões foram perfeitas (Rafael C. e Paloma S., 4/4 e 5/5) e a pior foi a de Valério A. (4/6, 67%). A dispersão entre voluntários é considerável, 11,4 pontos percentuais de desvio padrão, o que é esperado com apenas 8 sessões e listas de tarefas de tamanhos diferentes: uma falha isolada em uma sessão de 4 tarefas custa 25 pontos.

Importa mais o **tipo** de erro do que a contagem. As respostas abertas identificam de onde vieram as sete falhas:

- **Enquadramento incompleto:** Willian T. registrou que a "nota precisa ser colocada por completo na câmera". A localização clássica trabalha sobre o contorno do objeto claro no fundo escuro, então uma nota parcialmente fora do campo produz um recorte que a rede não reconhece.
- **Cédula fora do conjunto de treino:** Rafael C. apontou a impossibilidade de detectar notas de 200 reais, que estão fora do modelo atual por decisão explícita do roteiro.
- **Múltiplas notas simultâneas:** Paloma S. colocou duas cédulas juntas na bancada, situação não prevista pelo pipeline, que assume uma cédula por quadro.

Nenhum voluntário relatou que o sistema **falou um valor errado**. Esse é o resultado mais relevante da análise: a decisão de projeto registrada na modelagem funcional, de preferir o silêncio ao palpite, se sustentou na prática. As falhas observadas foram de omissão, não de confusão entre denominações, e o custo de uma omissão para a usuária (reposicionar a nota) é muito menor que o de um valor errado anunciado no caixa.

### 3.2 Tempo até a resposta

O tempo foi cronometrado do instante em que a cédula toca a bancada até o anúncio falado do valor, e cada voluntário tem aqui a média das suas tarefas. Ele mede a latência percebida, não a latência de inferência: inclui o tempo do voluntário acomodar a nota e o intervalo do laço de detecção até a estabilização.

In [ ]:
print(f"Tempo médio geral ....... {tempos.mean():.2f} s")
print(f"Desvio padrão ........... {tempos.std(ddof=1):.2f} s")
print(f"Mínimo e máximo ......... {tempos.min():.1f} s e {tempos.max():.1f} s")

barras(
    tempos,
    nomes,
    "Tempo médio até a resposta falada, por voluntário (segundos)",
    limite=3.6,
    formatar=lambda v: f"{v:.1f} s",
    cor=VERDE,
    referencia=(f"média {tempos.mean():.1f} s", tempos.mean()),
)
plt.show()

**Leitura.** A latência percebida foi de **2,81 segundos em média**, com todas as sessões entre 2,6 e 3,1 segundos. O desvio padrão de 0,16 s é pequeno: o sistema é **previsível**, e previsibilidade importa mais que velocidade bruta em uma interface sem retorno visual, porque é o que permite à usuária saber quando desistir e reposicionar a nota. Nenhum voluntário citou lentidão em Q13 (o que menos gostou), e três citaram a rapidez espontaneamente em Q12 ("velocidade de resposta", "muito rápido", "agilidade e precisão"), o que indica que a faixa de 3 segundos está dentro do que se considera resposta imediata para esta tarefa.

## 4. Usabilidade Percebida (SUS)

As questões Q1 a Q10 formam o **System Usability Scale**, aplicado na sua forma padrão: itens ímpares com afirmação positiva, itens pares com afirmação negativa, resposta de 1 (discordo totalmente) a 5 (concordo totalmente). A pontuação de cada voluntário é obtida somando `resposta - 1` nos itens positivos, `5 - resposta` nos negativos, e multiplicando o total por 2,5, o que produz um valor de 0 a 100. O valor **não** é uma porcentagem: a referência da literatura é que **68 corresponde à média** dos sistemas avaliados, e acima de 80 o sistema entra no quartil superior.

In [ ]:
ITENS_POSITIVOS = ["Q1", "Q3", "Q5", "Q7", "Q9"]
ITENS_NEGATIVOS = ["Q2", "Q4", "Q6", "Q8", "Q10"]


def pontuacao_sus(resposta):
    positivos = sum(int(resposta[q]) - 1 for q in ITENS_POSITIVOS)
    negativos = sum(5 - int(resposta[q]) for q in ITENS_NEGATIVOS)
    return 2.5 * (positivos + negativos)


sus = np.array([pontuacao_sus(r) for r in respostas])

print(f"SUS médio ............... {sus.mean():.1f}")
print(f"Desvio padrão ........... {sus.std(ddof=1):.1f}")
print(f"Mínimo e máximo ......... {sus.min():.1f} e {sus.max():.1f}")
print(f"Sessões acima de 68 ..... {(sus > 68).sum()} de {len(sus)}")

barras(
    sus,
    nomes,
    "Pontuação SUS por voluntário (0 a 100)",
    limite=115,
    formatar=lambda v: f"{v:.1f}",
    referencia=("média da literatura: 68", 68),
)
plt.show()

In [ ]:
# Contribuição de cada item do SUS, normalizada de 0 (pior) a 4 (melhor),
# para localizar quais afirmações puxaram a pontuação para baixo.
codigos = ITENS_POSITIVOS + ITENS_NEGATIVOS
codigos.sort(key=lambda c: int(c[1:]))

contribuicoes = []
for codigo in codigos:
    brutas = np.array([int(r[codigo]) for r in respostas])
    contribuicoes.append((brutas - 1 if codigo in ITENS_POSITIVOS else 5 - brutas).mean())
contribuicoes = np.array(contribuicoes)


def encurtar(texto, limite=46):
    return texto if len(texto) <= limite else texto[: limite - 1].rsplit(" ", 1)[0] + "..."


barras(
    contribuicoes,
    [f"{c}  {encurtar(questoes[c])}" for c in codigos],
    "Contribuição média por item do SUS (0 = pior, 4 = melhor)",
    limite=4.6,
    formatar=lambda v: f"{v:.2f}",
)
plt.show()

for codigo, valor in sorted(zip(codigos, contribuicoes), key=lambda p: p[1])[:3]:
    print(f"{codigo} ({valor:.2f}): {questoes[codigo]}")

In [ ]:
# Q11 não faz parte do SUS e é analisada isoladamente.
interatividade = np.array([int(r["Q11"]) for r in respostas])
print(f"{questoes['Q11']}")
print(f"Média: {interatividade.mean():.2f} de 5 | respostas: {np.bincount(interatividade, minlength=6)[1:]}")

**Leitura.** O SUS médio foi **96,6**, com todas as oito sessões acima de 87 e quatro delas com pontuação máxima. Está muito acima da média de referência (68) e coerente com o desenho do teste: a interação tem um único gesto (encostar a nota na bancada) e o sistema já chegava configurado e rodando, conforme o roteiro. A leitura honesta é que a escala está **saturada**: com um sistema de um gesto só, o SUS tem pouca margem para discriminar, e o valor confirma a ausência de atrito, sem servir como medida fina de qualidade.

Os três itens mais baixos são os informativos:

- **Q10** ("precisei aprender muitas coisas antes de usar", 3,50) empata como pior item, puxado por uma única resposta 5 de Samira H., isolada em meio a respostas 1. Como a mesma voluntária deu notas altas em todo o resto e escreveu "amei tudo" em Q13, é provável que seja inversão de escala no preenchimento em papel, um risco conhecido de itens negativos em ficha impressa.
- **Q1** ("gostaria de usar com frequência", 3,50) é o outro item no fundo da lista, com 3 de Paloma S. e 4 de Valério A. e Samira H. Faz sentido: nenhum dos voluntários tem deficiência visual, então a utilidade percebida no dia a dia é naturalmente menor que a facilidade percebida. Vinicius C. verbalizou isso em Q13 e Q14 ("não acho que terá uma grande aplicabilidade", "bom projeto, mas não vejo quão útil"), ainda que tenha dado 5 em todos os itens positivos.
- **Q6** ("várias inconsistências no sistema", 3,75) recebeu 2 de Paloma S. e Willian T., exatamente os dois voluntários que esbarraram em limitações reais: duas notas ao mesmo tempo e nota parcialmente fora do campo. O item negativo capturou justamente as falhas que as métricas objetivas apontaram.

Os sete itens restantes ficaram na pontuação máxima ou muito perto dela, incluindo Q4 e Q8, que medem a necessidade de suporte técnico e a complicação de uso, ambos com 4,00. Q11, fora do SUS, teve 5 de todos os oito voluntários.

## 5. Filmagens

Conforme previsto nas condições do roteiro, as sessões foram gravadas em vídeo com autorização dos voluntários, enquadrando a bancada e a tela e não o rosto do participante. A gravação consolidada dos testes está disponível em:

[Vídeo dos testes voluntários (Google Drive)](https://drive.google.com/file/d/1yOA3MfrLPN_YoBceT549B1oJ8bXyz385/view)

## 6. Coleta de Feedback

As questões abertas Q12 a Q14 coletam a impressão livre e as sugestões. As células abaixo listam as respostas na íntegra, sem edição, e agrupam os temas recorrentes.

In [ ]:
ABERTAS = [
    ("Q12 - Mais gostou", "Q12"),
    ("Q13 - Menos gostou", "Q13"),
    ("Q14 - Sugestões e comentários", "Q14"),
]

for coluna, codigo in ABERTAS:
    preenchidas = [(r["Nome completo"], r[coluna].strip()) for r in respostas if r[coluna].strip()]
    print(f"\n{codigo}. {questoes[codigo]}  ({len(preenchidas)}/{len(respostas)} responderam)")
    print("-" * 78)
    for nome, texto in preenchidas:
        print(f"  {nome:<12} {texto}")

In [ ]:
# Agrupamento manual dos temas recorrentes nas respostas abertas.
TEMAS = {
    "Elogio: velocidade da resposta": ["Raphael M.", "Luana R.", "Valério A.", "Samira H."],
    "Elogio: anúncio falado do valor": ["Rafael C.", "Vinicius C."],
    "Elogio: robustez (nota amassada, dobrada, ângulos)": ["Valério A.", "Willian T.", "Samira H."],
    "Elogio: rejeita o que não é cédula": ["Luana R."],
    "Pedido: somar ou contar várias notas de uma vez": ["Paloma S.", "Willian T.", "Samira H."],
    "Pedido: cobrir a cédula de 200 reais": ["Rafael C."],
    "Pedido: aceitar nota parcialmente no campo": ["Willian T."],
    "Pedido: integrar com outros sistemas": ["Valério A."],
    "Dúvida sobre a aplicabilidade do sistema": ["Vinicius C."],
}

for tema, citaram in sorted(TEMAS.items(), key=lambda p: -len(p[1])):
    print(f"{len(citaram)}x  {tema:<52} {', '.join(citaram)}")

**Leitura.** O feedback aberto converge em poucos pontos.

No lado positivo, os dois atributos mais citados são exatamente os dois que a modelagem funcional colocou como centrais: a **velocidade da resposta** (4 menções) e o **anúncio falado** do valor (2 menções, com Vinicius C. chamando a fala de "o diferencial"). A **robustez** apareceu de forma não solicitada em três fichas, sempre associada às tarefas mais difíceis do roteiro: nota amassada, nota dobrada e ângulos variados. Luana R. destacou que o sistema "não reconhece fotos", ou seja, notou e valorizou o comportamento de rejeição da tarefa T9, que era uma das hipóteses de risco do roteiro.

No lado negativo, a crítica dominante não é sobre qualidade e sim sobre **alcance**: três voluntários pediram, de forma independente, a contagem ou soma de várias cédulas de uma vez. Isso é uma extensão de escopo, não um defeito, mas é significativo que tenha sido o pedido mais frequente, porque corresponde ao uso real de conferir um maço de notas em vez de uma nota isolada. Os demais pedidos são limitações já conhecidas e documentadas: a cédula de 200 reais fora do dataset e a exigência de a nota estar inteira no campo.

A observação de Vinicius C. sobre aplicabilidade merece registro por ser a única voz destoante. Ela é consistente com o perfil dos voluntários, todos videntes, e reforça a limitação metodológica discutida na seção 8: o público-alvo do sistema não estava representado na amostra.

### 6.1 Compreensão do sistema pelos voluntários

As questões Q15 a Q17 pedem que o voluntário descreva, com as próprias palavras, o objetivo do sistema, os experimentos que fez e os resultados que obteve. Elas funcionam como uma verificação independente: se a descrição do voluntário coincide com o que o sistema realmente faz, a interação se explicou sozinha, sem tutorial.

In [ ]:
COMPREENSAO = [
    ("Q15 - Objetivo do sistema", "Q15"),
    ("Q16 - Experimentos realizados", "Q16"),
    ("Q17 - Resultados obtidos", "Q17"),
]

for coluna, codigo in COMPREENSAO:
    print(f"\n{codigo}. {questoes[codigo]}")
    print("-" * 78)
    for r in respostas:
        print(f"  {r['Nome completo']:<12} {r[coluna].strip()}")

# Um voluntário "acertou o objetivo" se citou reconhecer, identificar ou ler o valor de cédulas.
PALAVRAS = ("cédula", "nota", "dinheiro")
acertaram = [r["Nome completo"] for r in respostas if any(p in r["Q15 - Objetivo do sistema"].lower() for p in PALAVRAS)]
print(f"\nDescreveram o objetivo em termos de cédulas/dinheiro: {len(acertaram)}/{len(respostas)}")

**Leitura.** Sete dos oito voluntários descreveram o objetivo do sistema exatamente como ele foi projetado, em termos de identificar ou reconhecer o valor de cédulas. A oitava resposta, de Paloma S., foi "acessibilidade", que não cita cédulas mas identifica a **finalidade** do sistema, e Luana R. chegou à mesma leitura sem ter sido informada ("muito interessante para pessoas com deficiência visual"). Ou seja, duas das oito pessoas inferiram o propósito de acessibilidade apenas usando o sistema, o que é um bom sinal para a comunicação da proposta.

As descrições de experimento em Q16 confirmam que o roteiro foi seguido (várias cédulas, uma de cada vez, denominações diferentes, notas dobradas e em ângulos), e as de Q17 são uniformemente positivas, com as ressalvas já contabilizadas nas métricas objetivas. Vale notar a formulação de Samira H., "reconheceu quase todas as notas, em diferentes ângulos e posições", que é uma descrição mais fiel dos 83% de acerto global do que as respostas que dizem "todas".

## 7. Confronto com as Metas da Modelagem Funcional

A modelagem funcional (etapa 3) associou uma métrica objetiva a cada parte do sistema. A tabela abaixo confronta o que foi prometido com o que os testes voluntários mediram.

In [ ]:
resumo = [
    ("Taxa de acerto no uso real", f"{taxa_global:.1%} ({acertos.sum()}/{tarefas.sum()} tarefas)", "Atendido"),
    ("Ausência de valor falado errado", "0 ocorrências relatadas", "Atendido"),
    ("Latência percebida", f"{tempos.mean():.2f} s (dp {tempos.std(ddof=1):.2f} s)", "Atendido"),
    ("Usabilidade sem treinamento", f"SUS {sus.mean():.1f} de 100", "Atendido"),
    ("Interatividade percebida (Q11)", f"{interatividade.mean():.2f} de 5", "Atendido"),
    ("Cobertura de denominações", "6 de 7 (falta a de 200 reais)", "Parcial"),
    ("Múltiplas cédulas simultâneas", "Fora do escopo atual", "Não atendido"),
]

largura = max(len(item) for item, _, _ in resumo)
print(f"{'Meta':<{largura}}  {'Resultado medido':<32}  Situação")
print("-" * (largura + 46))
for item, valor, situacao in resumo:
    print(f"{item:<{largura}}  {valor:<32}  {situacao}")

## 8. Conclusões e Próximos Passos

Os testes com voluntários confirmam a hipótese central do projeto: **o sistema é utilizável sem instrução prévia e é conservador nos seus erros**. Uma pessoa que nunca viu o programa coloca a nota na bancada, ouve o valor em menos de 3 segundos e descreve corretamente o que o sistema faz. Nas 42 tarefas executadas, as 7 falhas foram todas de omissão, com causas identificadas e nenhuma confusão entre denominações, que era o risco grave mapeado na modelagem funcional.

As três frentes de trabalho que os resultados apontam, em ordem de retorno:

1. **Contagem de múltiplas cédulas.** É o pedido mais frequente do feedback (3 de 8 voluntários) e a única falha estrutural observada. Exige tratar mais de um contorno por quadro na localização e acumular os valores anunciados, sem mudar o modelo de classificação.
2. **Cobertura da cédula de 200 reais.** Limitação conhecida e de solução direta: coletar e rotular exemplares, retreinar a rede e revalidar a matriz de confusão.
3. **Retorno de posicionamento.** O caso da nota parcialmente fora do campo hoje resulta em silêncio, indistinguível para a usuária de "não reconheci". Um aviso falado curto para bancada vazia ou cédula cortada resolveria o problema apontado por Willian T. e é o ajuste mais relevante para o público que depende só do áudio.

Fica também uma correção de método para uma eventual próxima rodada de testes: recrutar pelo menos um participante com deficiência visual e aplicar a enquete em formato digital, com a escala rotulada item a item.